# Notebook 5 — Iterative Capacity Probing

**Course: Cross-Corridor Capacity Analysis with pandapower (Svedala grid)**

Now we automate. From `ΔP = 0` we step the requested transfer up by a fixed
increment, run the power flow at each step, and stop when the first operating
limit is violated (or the power flow diverges). The last *feasible* corridor
flow is the **Total Transfer Capacity (TTC)** for the corridor in the chosen
direction, against the chosen base case.

## Learning objectives

- Build a coarse linear sweep that records every step.
- Detect the first limit violation and the **limiting element**.
- Refine with **bisection** to ±1 MW resolution.
- Repeat for the reverse direction.
- Map the result onto the TSO vocabulary: TTC → NTC.

## 5.1  Set up

In [ ]:
import json, copy
import pandapower as pp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

net = pp.from_json('data/svedala_base.json')
pp.runpp(net)

with open('data/corridor.json') as f:
    corridor = json.load(f)
GSK_A = pd.Series({int(k): float(v) for k, v in corridor['gsk_a'].items()})
GSK_B = pd.Series({int(k): float(v) for k, v in corridor['gsk_b'].items()})
LINE_DIR  = [tuple(x) for x in corridor['corridor_lines']]
TRAFO_DIR = [tuple(x) for x in corridor['corridor_trafos']]
P0        = corridor['base_flow_a_to_b']
ZONE_SOURCE = corridor['zone_source']
ZONE_SINK   = corridor['zone_sink']
print(f'Base flow {ZONE_SOURCE} → {ZONE_SINK}: {P0:+.2f} MW')

## 5.2  Helpers

In [ ]:
def corridor_flow(net, line_dir, trafo_dir):
    p = 0.0
    for idx, d in line_dir:  p += d * net.res_line.at[idx, 'p_from_mw']
    for idx, d in trafo_dir: p += d * net.res_trafo.at[idx, 'p_hv_mw']
    return p

def is_feasible(net, v_min=0.95, v_max=1.05,
                line_limit=100.0, trafo_limit=100.0):
    if (net.res_line.loading_percent  > line_limit ).any(): return False
    if len(net.res_trafo) and (net.res_trafo.loading_percent > trafo_limit).any(): return False
    if (net.res_bus.vm_pu < v_min).any(): return False
    if (net.res_bus.vm_pu > v_max).any(): return False
    return True

def limiting_element(net, v_min=0.95, v_max=1.05,
                     line_limit=100.0, trafo_limit=100.0):
    candidates = []
    if len(net.res_line):
        i = net.res_line.loading_percent.idxmax()
        candidates.append(('line', int(i),
                          float(net.res_line.loading_percent[i] / line_limit)))
    if len(net.res_trafo):
        i = net.res_trafo.loading_percent.idxmax()
        candidates.append(('trafo', int(i),
                          float(net.res_trafo.loading_percent[i] / trafo_limit)))
    i = net.res_bus.vm_pu.idxmin()
    if net.res_bus.vm_pu[i] < v_min:
        candidates.append(('vlow', int(i), float(v_min / net.res_bus.vm_pu[i])))
    i = net.res_bus.vm_pu.idxmax()
    if net.res_bus.vm_pu[i] > v_max:
        candidates.append(('vhigh', int(i), float(net.res_bus.vm_pu[i] / v_max)))
    return max(candidates, key=lambda c: c[2])

def apply_dispatch_shift(net, delta_mw, gsk_a, gsk_b):
    for g, k in gsk_a.items():
        new_p = net.gen.at[g, 'p_mw'] + k * delta_mw
        if not np.isnan(net.gen.at[g, 'max_p_mw']): new_p = min(new_p, net.gen.at[g, 'max_p_mw'])
        if not np.isnan(net.gen.at[g, 'min_p_mw']): new_p = max(new_p, net.gen.at[g, 'min_p_mw'])
        net.gen.at[g, 'p_mw'] = new_p
    for g, k in gsk_b.items():
        new_p = net.gen.at[g, 'p_mw'] - k * delta_mw
        if not np.isnan(net.gen.at[g, 'max_p_mw']): new_p = min(new_p, net.gen.at[g, 'max_p_mw'])
        if not np.isnan(net.gen.at[g, 'min_p_mw']): new_p = max(new_p, net.gen.at[g, 'min_p_mw'])
        net.gen.at[g, 'p_mw'] = new_p

## 5.3  Coarse sweep

In [ ]:
def coarse_sweep(base_net, gsk_a, gsk_b, line_dir, trafo_dir,
                 step_mw=50.0, max_delta_mw=3000.0,
                 **kw):
    rows = []
    delta = 0.0
    while delta <= max_delta_mw:
        net2 = copy.deepcopy(base_net)
        apply_dispatch_shift(net2, delta, gsk_a, gsk_b)
        try:
            pp.runpp(net2)
            converged = True
        except pp.LoadflowNotConverged:
            rows.append({'delta_mw': delta, 'converged': False,
                         'corridor_flow': None, 'feasible': False,
                         'limiting_kind': 'NON_CONVERGED', 'limiting_idx': -1})
            break
        feas = is_feasible(net2, **kw)
        kind, idx, _ = limiting_element(net2, **kw) if not feas else ('-', -1, 0.0)
        rows.append({
            'delta_mw':         delta,
            'converged':        True,
            'corridor_flow':    corridor_flow(net2, line_dir, trafo_dir),
            'max_line_loading': float(net2.res_line.loading_percent.max()),
            'min_voltage':      float(net2.res_bus.vm_pu.min()),
            'max_voltage':      float(net2.res_bus.vm_pu.max()),
            'feasible':         feas,
            'limiting_kind':    kind,
            'limiting_idx':     idx,
        })
        if not feas: break
        delta += step_mw
    return pd.DataFrame(rows)

sweep = coarse_sweep(net, GSK_A, GSK_B, LINE_DIR, TRAFO_DIR,
                     step_mw=50.0, max_delta_mw=2500.0)
sweep.tail(8).round(3)

## 5.4  Coarse TTC

In [ ]:
feas = sweep[sweep.feasible]
infeas = sweep[~sweep.feasible]
ttc_lo = feas.corridor_flow.iloc[-1]
ttc_hi = (infeas.corridor_flow.iloc[0]
          if len(infeas) and infeas.corridor_flow.iloc[0] is not None
          else float('nan'))
print(f'Coarse TTC {ZONE_SOURCE} → {ZONE_SINK}:')
print(f'  lower bound: {ttc_lo:+.2f} MW (last feasible)')
print(f'  upper bound: {ttc_hi:+.2f} MW (first infeasible)')
if len(infeas):
    fb = infeas.iloc[0]
    print(f'  limiting element at first violation: {fb.limiting_kind} #{int(fb.limiting_idx)}')

## 5.5  Plot

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
x = sweep.corridor_flow
ax = axes[0]
ax.plot(x, sweep.max_line_loading, 'o-', label='max line loading')
ax.axhline(100, color='red', linestyle='--', label='100 % limit')
ax.set_ylabel('Loading [%]')
ax.set_title(f'Capacity probe {ZONE_SOURCE} → {ZONE_SINK}')
ax.legend(); ax.grid(True, alpha=0.4)
ax = axes[1]
ax.plot(x, sweep.min_voltage, 'o-', label='min voltage')
ax.plot(x, sweep.max_voltage, 's-', label='max voltage')
ax.axhline(0.95, color='red', linestyle='--')
ax.axhline(1.05, color='red', linestyle='--')
ax.set_xlabel(f'Corridor flow {ZONE_SOURCE} → {ZONE_SINK} [MW]')
ax.set_ylabel('Voltage [p.u.]')
ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

## 5.6  Bisection refinement

In [ ]:
def bisect_ttc(base_net, gsk_a, gsk_b, line_dir, trafo_dir,
               lo_delta, hi_delta, tol_mw=1.0, max_iter=30, **kw):
    last_flow = None
    last_delta = lo_delta
    last_limiting = ('-', -1)
    for _ in range(max_iter):
        if hi_delta - lo_delta <= tol_mw:
            break
        mid = 0.5 * (lo_delta + hi_delta)
        net2 = copy.deepcopy(base_net)
        apply_dispatch_shift(net2, mid, gsk_a, gsk_b)
        try:
            pp.runpp(net2)
            if is_feasible(net2, **kw):
                last_flow = corridor_flow(net2, line_dir, trafo_dir)
                last_delta = mid
                lo_delta = mid
            else:
                k, i, _ = limiting_element(net2, **kw)
                last_limiting = (k, i)
                hi_delta = mid
        except pp.LoadflowNotConverged:
            last_limiting = ('NON_CONVERGED', -1)
            hi_delta = mid
    return {'ttc_delta_mw':   last_delta,
            'ttc_flow_mw':    last_flow,
            'limiting_kind':  last_limiting[0],
            'limiting_idx':   last_limiting[1]}

lo = float(feas.delta_mw.iloc[-1])
hi = float(infeas.delta_mw.iloc[0]) if len(infeas) else lo + 50.0
ttc = bisect_ttc(net, GSK_A, GSK_B, LINE_DIR, TRAFO_DIR, lo, hi)
print('Refined TTC:')
for k, v in ttc.items():
    print(f'  {k:<16} {v}')

## 5.7  Reverse direction

In [ ]:
def coarse_sweep_negative(base_net, gsk_a, gsk_b, line_dir, trafo_dir,
                          step_mw=-50.0, min_delta_mw=-3000.0, **kw):
    rows = []
    delta = 0.0
    while delta >= min_delta_mw:
        net2 = copy.deepcopy(base_net)
        apply_dispatch_shift(net2, delta, gsk_a, gsk_b)
        try:
            pp.runpp(net2)
        except pp.LoadflowNotConverged:
            rows.append({'delta_mw': delta, 'corridor_flow': None,
                         'feasible': False}); break
        feas = is_feasible(net2, **kw)
        rows.append({'delta_mw': delta,
                     'corridor_flow': corridor_flow(net2, line_dir, trafo_dir),
                     'max_line_loading': float(net2.res_line.loading_percent.max()),
                     'min_voltage': float(net2.res_bus.vm_pu.min()),
                     'max_voltage': float(net2.res_bus.vm_pu.max()),
                     'feasible': feas})
        if not feas: break
        delta += step_mw
    return pd.DataFrame(rows)

sweep_neg = coarse_sweep_negative(net, GSK_A, GSK_B, LINE_DIR, TRAFO_DIR,
                                  step_mw=-50.0, min_delta_mw=-2500.0)
feas_neg = sweep_neg[sweep_neg.feasible]
if len(feas_neg):
    print(f'Coarse TTC {ZONE_SINK} → {ZONE_SOURCE} (positive magnitude):')
    print(f'  lower bound: {-feas_neg.corridor_flow.iloc[-1]:+.2f} MW')

## 5.8  TTC, TRM, NTC

What we computed is **TTC** — the maximum transfer the grid physically
supports against the *base case* in steady state. To translate it into the
**NTC** offered to the day-ahead market, TSOs subtract a **TRM** (Transmission
Reliability Margin) that covers measurement errors, unforeseen flows and (most
importantly) the **N-1 criterion**:

$$ \text{NTC} = \text{TTC} - \text{TRM} $$

Combining the capacity sweep with N-1 contingency analysis gives the N-1-secure
TTC directly. That combination is the topic of the **combined assignment**
(`n1_capacity_assignment/`).

## 5.9  Save

In [ ]:
sweep.to_csv('data/capacity_sweep_pos.csv', index=False)
sweep_neg.to_csv('data/capacity_sweep_neg.csv', index=False)
with open('data/ttc_pos.json', 'w') as f:
    json.dump(ttc, f, indent=2)
print('Saved sweeps and TTC to data/')

## 5.10  Exercises

1. Run the sweep with a *finer* step (10 MW). How much does the TTC lower
   bound improve, and at what cost in PF runs?
2. The slack lives in `ZON_EXTERN`. Investigate: how much does the slack
   active power change between `ΔP = 0` and `ΔP = TTC`? This tells you how
   much *extra* power the EXTERN zone implicitly provides — it's a cousin of
   the TRM.
3. Write a function `bisect_n1_ttc()` (a small change to `bisect_ttc`) that
   also runs an N-1 screen at every midpoint and only accepts the midpoint as
   feasible if **all contingencies** are feasible. This is the bridge to the
   combined assignment.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3


---

✅ **Checkpoint reached.** TTC computed in both directions.

Continue to [Notebook 6 — Reporting & Visualization](06_reporting.ipynb).